# YOLO11 warna + mapping robot: ICAM-300 / SmoothingPitch

Training di **Google Colab A100**, deployment deteksi di **ICAM-300**.
Model ringan `yolo11n-seg.pt` dipertahankan karena sudah berhasil pada perangkat Anda.
Notebook menggabungkan dua ZIP YOLO **segmentation**, dengan ID final tetap
**0=GREEN, 1=YELLOW, 2=RED** agar cocok dengan model/runtime lama. Dataset baru memakai
0=GREEN, 1=RED, 2=YELLOW: ID 1 dan 2 dipetakan ulang pada salinan label.

Alur: pilih dataset → audit → train/finetune/evaluate → evaluasi PT dan ONNX →
pilih confidence dari validation → pratinjau mapping → bundle dengan checksum.
Semua cell dapat Run All **tanpa mengirim gerakan robot**.

Dataset lama berisi 3.807 crop satu objek (3.480 train, 233 valid, 94 test), tanpa negatif.
Dataset baru arm_pasti_3_new.v1i.yolov11.zip berisi 568 gambar (525 train, 27 valid, 16 test),
sebagian besar multiobjek. Sebelum deduplikasi, gabungan berisi 4.375 gambar:
4.005 train, 260 valid, 110 test. Kedua dataset belum memiliki contoh negatif berlabel kosong.
Matras hijau pada dataset baru adalah **bidang kerja/mapping**, bukan kelas GREEN tambahan.
GREEN berarti objek hijau yang dianotasi. Jangan menyaring seluruh warna hijau dari gambar.
Ambil frame penuh, beberapa objek per warna, meja kosong, tangan/robot, bayangan dan pantulan.
Pisahkan sesi pengambilan sebelum augmentasi. Kamera, pencahayaan, dan bidang ukur harus tetap.

Firmware acuan mapping sekarang **`SmoothingPitch.ino`** (5 stepper + servo):
`X Y Z Pitch DOF4 Servo` untuk PTP, tambah `Travel H` positif untuk parabolik.
Koordinat warna Anda disimpan sebagai **titik referensi**; peran ambil/letak belum ditentukan.
Vision menghasilkan mask/centroid piksel, bukan kedalaman atau lintasan pick-and-place.

ICAM-300 menggunakan Qualcomm QCS6490. Runtime ini **ONNX Runtime CPU**;
latency Colab bukan FPS ICAM. NPU/QNN memerlukan integrasi terpisah.

Mekanisme sorting: satu stage memilih satu benda, mengambil, meletakkan ke mangkok warna
yang ditentukan Arduino, lalu HOME. Selama stage berjalan target lain tidak diantrekan;
pemilihan berikutnya menggunakan frame baru setelah status selesai/HOME dari Arduino.


## 1. Persiapan Google Colab A100

1. Upload notebook ini ke Google Colab.
2. Pilih **Runtime → Change runtime type → A100 GPU**, lalu hubungkan runtime.
3. Simpan KEDUA ZIP di /content atau MyDrive/arm-color-yolo/datasets/.
   Cell pemilihan meminta upload satu per satu hanya untuk ZIP yang belum tersedia.
   ZIP_SOURCES berisi dua nama default; bisa diganti path absolut Google Drive.
4. Jalankan cell berurutan. Cell penyimpanan meminta akses Drive melalui UI Colab.

Default A100: **batch 64, image 512, maksimum 150 epoch (patience 30), AMP aktif**; workers mengikuti CPU runtime (maksimal 4).
GPU yang terpasang diperiksa oleh kode. Model tetap YOLO11n-seg agar deployment ICAM ringan.
Jika mengganti model/ukuran gambar dan mengalami CUDA out-of-memory, turunkan `BATCH` ke 32 atau 16.
Tidak perlu API key Roboflow. Instalasi berikut khusus mesin training dan memakai PyTorch bawaan Colab.
Dependensi training mendukung **Python 3.10–3.13**, termasuk Python 3.13 pada log Colab Anda:
NumPy 2.2.6 dan ONNX 1.18.0 memiliki wheel Python 3.13. `--only-binary=:all:` mencegah
pip mencoba kompilasi source. Pin lama NumPy 1.26.4 / ONNX 1.17.0 tidak dipakai untuk Colab.

**Setelah cell instalasi berhasil, pilih Runtime → Restart session satu kali**, lalu lanjutkan
dari cell pemeriksaan environment. Ini membersihkan modul versi lama yang mungkin sudah dimuat.
Tidak perlu menjalankan ulang instalasi setelah restart. Jangan pilih Disconnect and delete runtime
karena file `/content` akan hilang. Jika ada pesan `ERROR`, jangan lanjut ke training.


In [ ]:
%pip install --only-binary=:all: "ultralytics==8.3.221" "numpy==2.2.6" "opencv-python==4.11.0.86" "PyYAML==6.0.2" "onnx==1.18.0" "onnxruntime==1.20.1" "pyserial==3.5"

# Setelah instalasi berhasil, restart sekali melalui Runtime > Restart session.
# Setelah tersambung kembali, lanjutkan ke pemeriksaan environment; jangan ulangi instalasi.

In [ ]:
# Jalankan setelah Restart session. Cek import/ABI sebelum memakai waktu A100 untuk training.
import sys
from importlib.metadata import version

print('Python:', sys.version.split()[0])
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Dependensi notebook ini disiapkan untuk Python 3.10–3.13.')
for package in ('numpy', 'onnx', 'onnxruntime'):
    loaded = sys.modules.get(package)
    if loaded is not None and getattr(loaded, '__version__', None) != version(package):
        raise RuntimeError(f'{package} versi lama masih dimuat. Runtime > Restart session, lalu ulangi cell ini.')

import numpy as np
import cv2
import onnx
import onnxruntime as ort
import torch
from ultralytics import YOLO

for package in ('ultralytics', 'numpy', 'opencv-python', 'onnx', 'onnxruntime', 'torch', 'torchvision'):
    print(package, version(package))

# Model Identity kecil: memverifikasi bahwa NumPy, ONNX, ORT dan PyTorch dapat bertukar tensor.
probe = np.array([[1.0, 2.0, 3.0]], dtype=np.float32)
graph = onnx.helper.make_graph(
    [onnx.helper.make_node('Identity', ['x'], ['y'])], 'environment_check',
    [onnx.helper.make_tensor_value_info('x', onnx.TensorProto.FLOAT, [1, 3])],
    [onnx.helper.make_tensor_value_info('y', onnx.TensorProto.FLOAT, [1, 3])],
)
probe_model = onnx.helper.make_model(graph, opset_imports=[onnx.helper.make_opsetid('', 13)], ir_version=8)
onnx.checker.check_model(probe_model)
probe_session = ort.InferenceSession(probe_model.SerializeToString(), providers=['CPUExecutionProvider'])
np.testing.assert_allclose(probe_session.run(None, {'x': probe})[0], probe)
np.testing.assert_allclose(torch.from_numpy(probe).numpy(), probe)
assert cv2.resize(np.zeros((4, 4, 3), dtype=np.uint8), (2, 2)).shape == (2, 2, 3)
del probe_session, probe_model, graph, probe
print('Environment OK. Lanjutkan ke mount Drive dan konfigurasi A100.')

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
ROOT = Path('/content') if IN_COLAB else Path.cwd()
os.chdir(ROOT)
WORK = ROOT/'arm_color_work'  # Ekstraksi gambar tetap pada disk lokal runtime.
WORK.mkdir(parents=True, exist_ok=True)
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    PERSIST = Path('/content/drive/MyDrive/arm-color-yolo')
else:
    PERSIST = WORK
    print('Drive tidak aktif: unduh hasil sebelum runtime berakhir.')
RUNS_ROOT = PERSIST/'runs'
ARTIFACT_ROOT = PERSIST/'artifacts'
for folder in (PERSIST/'datasets', RUNS_ROOT, ARTIFACT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', RUNS_ROOT)
print('Model/bundle:', ARTIFACT_ROOT)

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import os
import shutil
import warnings
import zipfile
import cv2
import numpy as np
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, Image

MODEL = 'yolo11n-seg.pt'
IMGSZ = 512  # Pertahankan ukuran yang sudah teruji di ICAM; runtime memakai stretch.
EPOCHS = 150  # Batas atas; early stopping memilih best.pt, bukan epoch terakhir.
REQUIRE_GPU = IN_COLAB
HAS_CUDA = torch.cuda.is_available()
if REQUIRE_GPU and not HAS_CUDA:
    raise RuntimeError('GPU belum aktif. Pilih Runtime > Change runtime type > A100 GPU.')
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'
VRAM_GB = torch.cuda.get_device_properties(0).total_memory/1024**3 if HAS_CUDA else 0
BATCH = 64 if 'A100' in GPU_NAME.upper() and VRAM_GB >= 32 else (16 if HAS_CUDA else 4)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = min(4, os.cpu_count() or 1) if IN_COLAB else 0
SEED = 42
CONF = 0.60  # Sesuaikan dari precision/recall valid + rekaman meja penuh.
RESUME_CHECKPOINT = None  # Contoh: RUNS_ROOT/'arm_colors/weights/last.pt' dari sesi terputus.
print(f'GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GiB | batch: {BATCH} | workers: {WORKERS}')
print('Device:', DEVICE, '| dataset dipilih pada cell berikutnya.')
if HAS_CUDA and 'A100' not in GPU_NAME.upper():
    print('GPU yang tersedia bukan A100; batch 16 dipakai. Ubah jenis runtime jika ingin A100.')
if DEVICE == 'cpu':
    print('Training CPU bisa lama. Untuk cek alur dulu, set EPOCHS=1; training final gunakan GPU.')

# train: bobot pretrained; finetune: best.pt Anda + dataset baru; evaluate: tanpa training.
RUN_MODE = 'train'  # 'train', 'finetune', atau 'evaluate'
INITIAL_WEIGHTS = None  # Path best.pt lama, wajib untuk finetune/evaluate.
PATIENCE = 30
STRICT_SPLITS = True  # Hentikan training jika gambar identik lintas split.
TUNE_CONFIDENCE = True  # Validation saja; test tidak dipakai memilih threshold.
MIN_CLASS_PRECISION = 0.98

# 'last': confidence dari valid sumber terakhir (dataset matras baru).
# 'all': confidence dari valid gabungan yang didominasi crop lama.
CONFIDENCE_SOURCE = 'last'


## 2. Pilih kedua dataset

Default ZIP_SOURCES: dataset lama terlebih dahulu, dataset matras baru terakhir.
Path Windows tidak dapat dibaca Colab; upload kedua ZIP atau gunakan path Drive.
Untuk satu dataset saja, sisakan satu elemen list; ID final tetap GREEN/YELLOW/RED.
File ZIP harus berisi data.yaml, train, valid (atau val), test,
masing-masing dengan images/ dan labels/. Label harus poligon, bukan bbox 5 kolom.
Negatif menggunakan file label kosong. ZIP asli tidak diubah.

Jika ZIP belum ada, cell otomatis membuka dialog upload Colab meskipun upload_zip=None.
Pilih ZIP yang diminta, bukan notebook/model. Nama ZIP menggunakan titik biasa tanpa backslash.
Jika file sudah di Drive pada folder lain, isi path lengkap di ZIP_SOURCES.
Tidak perlu menjalankan ulang instalasi/training hanya untuk memperbaiki pemilihan ZIP.


In [ ]:
def resolve_dataset_zips(root, persist, sources, upload=None, allow_upload=True):
    root, persist = Path(root), Path(persist)
    if not sources or isinstance(sources, (str, Path)):
        raise ValueError('ZIP_SOURCES harus list nama/path ZIP, bukan satu string')
    resolved = []
    for source in sources:
        source_text = str(source)
        requested = Path(source_text).expanduser()
        candidates = [requested] if requested.is_absolute() else [root/requested, persist/'datasets'/requested]
        selected = next((p for p in candidates if p.is_file()), None)
        # Repair Markdown-style escaped dots only in a bare filename, never a directory path.
        cleaned = source_text.replace('\\.', '.')
        if selected is None and cleaned != source_text and not any(s in cleaned for s in ('/', '\\', ':')):
            corrected = [root/cleaned, persist/'datasets'/cleaned]
            candidates.extend(corrected)
            selected = next((p for p in corrected if p.is_file()), None)
            print(f'Nama ZIP dinormalisasi: {source_text!r} -> {cleaned!r}')
        if selected is None:
            # Lazy fallback also works when a copied cell left upload_zip=None or IN_COLAB=False.
            if upload is None and allow_upload:
                try:
                    from google.colab import files
                except ImportError:
                    pass
                else:
                    upload = files.upload
            if upload is None or not allow_upload:
                existing = []
                for directory in (root, persist/'datasets'):
                    if directory.is_dir():
                        existing.extend(str(p) for p in directory.iterdir()
                                        if p.is_file() and p.suffix.lower() == '.zip')
                locations = '\n  '.join(str(p) for p in candidates)
                available = '\n  '.join(sorted(set(existing))[:10]) or '(tidak ada)'
                raise FileNotFoundError(
                    f'ZIP belum ada: {source}\nDicari di:\n  {locations}\n'
                    f'ZIP yang tersedia:\n  {available}\n'
                    'Upload kedua ZIP ke /content atau MyDrive/arm-color-yolo/datasets/, '
                    'atau isi ZIP_SOURCES dengan path file yang tepat. '
                    'Di Colab, jalankan ulang seluruh cell ini untuk membuka dialog upload '
                    '(allow_upload harus True).')
            print('Pilih SATU file untuk sumber:', cleaned)
            uploaded = upload(target_dir=str(root))
            if len(uploaded) != 1:
                raise ValueError('Upload tepat satu ZIP yang diminta, lalu jalankan ulang cell ini')
            selected = Path(next(iter(uploaded)))
            if not selected.is_absolute():
                selected = root/selected
            del uploaded
        if not selected.is_file() or selected.suffix.lower() != '.zip' or not zipfile.is_zipfile(selected):
            raise ValueError(f'File bukan ZIP dataset yang valid: {selected}')
        if selected.parent.resolve() != root.resolve():
            digest = hashlib.sha256(selected.read_bytes()).hexdigest()[:12]
            local = root/'arm_color_work'/'archives'/digest/selected.name
            local.parent.mkdir(parents=True, exist_ok=True)
            if selected.resolve() != local.resolve():
                shutil.copy2(selected, local)
            selected = local
        resolved.append(selected.resolve())
    if len(set(resolved)) != len(resolved):
        raise ValueError('Path ZIP yang sama dipilih dua kali')
    return resolved


ZIP_SOURCES = [
    'arm-warna-3.v5i.yolov11.zip',
    'arm_pasti_3_new.v1i.yolov11.zip',  # Tidak ada backslash sebelum titik.
]  # Bisa path absolut Drive; untuk satu dataset, sisakan satu item.
# Fungsi membuka dialog upload Colab otomatis HANYA untuk ZIP yang belum ditemukan.
# Tidak bergantung pada flag IN_COLAB; upload_zip=None tetap mengaktifkan fallback ini.
upload_zip = None
ZIP_PATHS = resolve_dataset_zips(ROOT, PERSIST, ZIP_SOURCES, upload=upload_zip)
for selected in ZIP_PATHS:
    print(f'Sumber: {selected} ({selected.stat().st_size / 1024**2:.1f} MiB)')



### Ekstraksi, remapping, dan penggabungan split

Dataset lama: 0 HIJAU→GREEN, 1 KUNING→YELLOW, 2 MERAH→RED.
Dataset baru: 0 GREEN→0, **1 RED→2, 2 YELLOW→1**.
Koordinat poligon dipertahankan. Prefix sumber pada nama gambar/label mencegah benturan.

Train digabung dengan train, valid dengan valid, test dengan test; tidak mengacak ulang
split yang sudah ada. Duplikat piksel dalam split yang sama disimpan sekali hanya jika label
yang sudah dinormalisasi cocok. Gambar identik lintas split atau anotasi bertentangan
menghentikan merge. Nama foto asal lintas split juga dilaporkan sebelum prefix ditambahkan;
hash tidak dapat mendeteksi semua kebocoran frame berdekatan/sesi video.

merge_manifest.json menyimpan asal tiap gambar, remapping ID, SHA256 kedua ZIP dan atribusi
Roboflow/lisensi masing-masing. Identitas gabungan dipakai untuk memeriksa resume.


In [ ]:
def prepare_dataset(zip_path, work):
    digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()
    target = Path(work) / ('dataset_' + digest[:12])
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        seen = {}
        for info in archive.infolist():
            dest = (target / info.filename).resolve()
            if not dest.is_relative_to(target.resolve()):
                raise ValueError('Path ZIP keluar dari folder dataset')
            if info.is_dir():
                dest.mkdir(parents=True, exist_ok=True)
                continue
            payload = archive.read(info)
            fingerprint = hashlib.sha256(payload).hexdigest()
            if info.filename in seen and seen[info.filename] != fingerprint:
                raise ValueError('Entri ZIP duplikat berbeda: ' + info.filename)
            if info.filename not in seen:
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(payload)
                seen[info.filename] = fingerprint
    yamls = list(target.rglob('data.yaml'))
    if len(yamls) != 1:
        raise ValueError('Harus ada satu lokasi data.yaml')
    base = yamls[0].parent
    original = yaml.safe_load(yamls[0].read_text())
    original_names = original['names']
    if isinstance(original_names, list):
        original_names = dict(enumerate(original_names))
    aliases = {'HIJAU': 'GREEN', 'KUNING': 'YELLOW', 'MERAH': 'RED',
               'GREEN': 'GREEN', 'YELLOW': 'YELLOW', 'RED': 'RED'}
    names = {int(k): aliases[str(v).strip().upper()] for k, v in original_names.items()}
    if sorted(names) != [0, 1, 2] or set(names.values()) != {'RED', 'GREEN', 'YELLOW'}:
        raise ValueError('Dataset harus tepat tiga kelas warna')
    # Normalisasi val/ -> valid/ hanya pada salinan kerja; ZIP asli tidak disentuh.
    if not (base/'valid').exists() and (base/'val').is_dir():
        shutil.copytree(base/'val', base/'valid')
    fixed = dict(path=str(base.resolve()), train='train/images', val='valid/images',
                 test='test/images', nc=3, names=names)
    if 'roboflow' in original:
        fixed['roboflow'] = original['roboflow']  # Pertahankan atribusi dataset.
    output = base / 'data_colors.yaml'
    output.write_text(yaml.safe_dump(fixed, sort_keys=False), encoding='utf-8')
    return base, output, names, digest


def audit_dataset(base, names, strict=True):
    report, all_hashes, pixel_hashes, split_sources = {}, {}, {}, {}
    for split in ('train', 'valid', 'test'):
        images = sorted(p for p in (base/split/'images').iterdir()
                        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
        labels = {p.stem: p for p in (base/split/'labels').glob('*.txt')}
        if not images or len({p.stem for p in images}) != len(images):
            raise ValueError(f'{split}: gambar kosong atau nama stem duplikat')
        if set(labels) != {p.stem for p in images}:
            raise ValueError(f'{split}: pasangan image/label tidak lengkap (negatif gunakan label kosong)')
        counts, instances, large, total, sources = Counter(), [], 0, 0, set()
        for image_path in images:
            image = cv2.imread(str(image_path))
            if image is None:
                raise ValueError(f'Gambar rusak: {image_path}')
            pixel_hash = hashlib.sha256(str(image.shape).encode()+image.tobytes()).hexdigest()
            pixel_hashes.setdefault(pixel_hash, set()).add(split)
            fingerprint = hashlib.sha256(image_path.read_bytes()).hexdigest()
            all_hashes.setdefault(fingerprint, set()).add(split)
            sources.add(image_path.stem.split('.rf.')[0])
            rows = labels[image_path.stem].read_text().splitlines()
            instance_count = 0
            for row in rows:
                if not row.strip():
                    continue
                values = np.asarray([float(v) for v in row.split()])
                if len(values) < 7 or (len(values)-1) % 2 or not np.isfinite(values).all():
                    raise ValueError(f'Butuh poligon YOLO, bukan bbox/classification: {image_path.name}')
                cid = int(values[0])
                if values[0] != cid or cid not in names:
                    raise ValueError('Class ID tidak valid')
                points = values[1:].reshape(-1, 2)
                if (points < 0).any() or (points > 1).any() or abs(cv2.contourArea(points.astype(np.float32))) <= 1e-8:
                    raise ValueError(f'Poligon tidak valid: {image_path.name}')
                counts[names[cid]] += 1
                instance_count += 1
                large += int(np.prod(np.ptp(points, axis=0)) > 0.8)
                total += 1
            instances.append(instance_count)
        if set(counts) != set(names.values()):
            raise ValueError(f'{split}: ada kelas yang tidak punya contoh')
        split_sources[split] = sources
        report[split] = dict(images=len(images), instances=total, per_class=dict(counts),
                             negative_images=instances.count(0), multi_object_images=sum(n>1 for n in instances),
                             large_bbox_fraction=large/max(total, 1))
    exact_overlap = sum(len(v)>1 for v in all_hashes.values())
    source_overlap = {f'{a}/{b}': len(split_sources[a] & split_sources[b])
                      for a, b in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]}
    decoded_overlap = sum(len(v)>1 for v in pixel_hashes.values())
    report['leakage'] = dict(exact_image_hashes=exact_overlap, decoded_pixel_hashes=decoded_overlap,
                             source_name_overlap=source_overlap)
    if strict and (exact_overlap or decoded_overlap):
        raise ValueError('Gambar identik lintas split: pisahkan gambar asal/sesi sebelum training')
    report['coverage'] = dict(
        all_splits_have_negatives=all(report[s]['negative_images'] > 0 for s in ('train', 'valid', 'test')),
        all_splits_have_multiobject=all(report[s]['multi_object_images'] > 0 for s in ('train', 'valid', 'test')),
        crop_dominated=report['train']['large_bbox_fraction'] > 0.8,
        limitation='Hash/nama tidak mendeteksi seluruh kebocoran frame video atau sesi berdekatan')
    if exact_overlap or any(source_overlap.values()):
        warnings.warn('Kemungkinan kebocoran split: pisahkan foto asal/sesi sebelum menilai mAP.')
    if report['train']['large_bbox_fraction'] > 0.8:
        warnings.warn('Dataset didominasi crop besar. Tambahkan scene meja penuh untuk lokalisasi multiobjek.')
    return report

def remap_polygon_labels(text, source_names, canonical_names):
    """Change class IDs only, preserving polygon coordinates and empty negative labels."""
    reverse = {name: cid for cid, name in canonical_names.items()}
    output = []
    for line in text.splitlines():
        tokens = line.split()
        if not tokens:
            continue
        values = np.asarray([float(v) for v in tokens])
        if len(values) < 7 or (len(values)-1) % 2 or not np.isfinite(values).all():
            raise ValueError('Merge memerlukan poligon YOLO finite, bukan bbox 5 kolom')
        source_id = int(values[0])
        if values[0] != source_id or source_id not in source_names:
            raise ValueError('Class ID sumber tidak valid')
        points = values[1:].reshape(-1, 2)
        if (points < 0).any() or (points > 1).any() or abs(cv2.contourArea(points.astype(np.float32))) <= 1e-8:
            raise ValueError('Poligon sumber tidak valid')
        output.append(' '.join([str(reverse[source_names[source_id]]), *tokens[1:]]))
    return '\n'.join(output)+ ('\n' if output else '')


def merge_datasets(zip_paths, work, strict=True):
    """Preserve splits; remap classes, deduplicate within split, reject leakage."""
    canonical = {0: 'GREEN', 1: 'YELLOW', 2: 'RED'}
    if not zip_paths:
        raise ValueError('Pilih minimal satu ZIP dataset')
    sources, records, decoded_seen, family_splits = [], [], {}, {}
    input_hashes = set()
    for number, zip_path in enumerate(zip_paths, 1):
        base, _, source_names, digest = prepare_dataset(zip_path, work)
        if digest in input_hashes:
            raise ValueError('ZIP yang sama dipilih dua kali')
        input_hashes.add(digest)
        audit = audit_dataset(base, source_names, strict=strict)
        source_id = f'source_{number:02d}'
        original_yaml = yaml.safe_load((base/'data.yaml').read_text(encoding='utf-8'))
        sources.append(dict(source_id=source_id, zip_name=Path(zip_path).name, sha256=digest,
                            original_names=source_names, canonical_names=canonical,
                            id_remap={k: next(i for i, name in canonical.items() if name == value)
                                      for k, value in source_names.items()},
                            attribution=original_yaml.get('roboflow', {}), audit=audit))
        for split in ('train', 'valid', 'test'):
            for image_path in sorted((base/split/'images').iterdir()):
                if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp'}:
                    continue
                label = base/split/'labels'/(image_path.stem+'.txt')
                remapped = remap_polygon_labels(label.read_text(), source_names, canonical)
                label_key = sorted(tuple(float(v) for v in row.split()) for row in remapped.splitlines())
                frame = cv2.imread(str(image_path))
                fingerprint = hashlib.sha256(str(frame.shape).encode()+frame.tobytes()).hexdigest()
                family = image_path.stem.split('.rf.')[0]
                family_splits.setdefault(family, set()).add(split)
                origin = dict(source_id=source_id, original_image=image_path.name, original_split=split)
                if fingerprint in decoded_seen:
                    previous = decoded_seen[fingerprint]
                    if previous['split'] != split:
                        raise ValueError(f'Gambar identik lintas split: {previous["origins"][0]} vs {origin}. '
                                         'Perbaiki pembagian sumber; merge tidak memindahkan test ke train.')
                    if previous['_label_key'] != label_key:
                        raise ValueError(f'Anotasi bertentangan pada gambar identik: {origin}')
                    previous['origins'].append(origin)
                    continue
                record = dict(split=split, image=f'{source_id}__{image_path.name}',
                              pixel_sha256=fingerprint, origins=[origin],
                              _source=image_path, _label=remapped, _label_key=label_key)
                records.append(record)
                decoded_seen[fingerprint] = record
    identity = dict(version=1, source_sha256=[s['sha256'] for s in sources],
                    canonical_names=canonical, split_policy='preserve', dedup='decoded_pixels_within_split')
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    target = Path(work)/('merged_'+digest[:16])
    expected_images = {target/r['split']/'images'/r['image'] for r in records}
    expected_labels = {target/r['split']/'labels'/(Path(r['image']).stem+'.txt') for r in records}
    for split in ('train', 'valid', 'test'):
        current_images = {p for p in (target/split/'images').glob('*') if p.is_file()}
        current_labels = set((target/split/'labels').glob('*.txt'))
        if current_images - expected_images or current_labels - expected_labels:
            raise ValueError(f'Folder merge berisi file tambahan: {target}; gunakan WORK baru')
        (target/split/'images').mkdir(parents=True, exist_ok=True)
        (target/split/'labels').mkdir(parents=True, exist_ok=True)
    for record in records:
        destination = target/record['split']/'images'/record['image']
        shutil.copy2(record['_source'], destination)
        (target/record['split']/'labels'/(destination.stem+'.txt')).write_text(record['_label'], encoding='utf-8')
    fixed = dict(path=str(target.resolve()), train='train/images', val='valid/images',
                 test='test/images', nc=3, names=canonical)
    output = target/'data_colors.yaml'
    output.write_text(yaml.safe_dump(fixed, sort_keys=False), encoding='utf-8')
    overlap = {key: sorted(splits) for key, splits in family_splits.items() if len(splits) > 1}
    if overlap:
        warnings.warn('Nama foto asal berulang lintas split/sumber; periksa merge_manifest sebelum training')
    provenance = [{key: value for key, value in record.items() if not key.startswith('_')} for record in records]
    manifest = dict(identity=identity, dataset_sha256=digest, sources=sources, records=provenance,
                    duplicates_removed=sum(len(r['origins'])-1 for r in records),
                    source_family_overlap=overlap,
                    merged_images_per_split=dict(Counter(r['split'] for r in records)))
    (target/'merge_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return target, output, canonical, digest, manifest


def records_for_source(records, provenance, source_id):
    """Split inference records by provenance, including retained exact duplicates."""
    names = {record['image'] for record in provenance
             if any(origin['source_id'] == source_id for origin in record['origins'])}
    return [record for record in records if Path(record['image']).name in names]


DATASET, DATA_YAML, NAMES, DATASET_SHA256, MERGE_MANIFEST = merge_datasets(ZIP_PATHS, WORK, strict=STRICT_SPLITS)
REPORT = audit_dataset(DATASET, NAMES, strict=STRICT_SPLITS)
REPORT['sources'] = [{k: v for k, v in source.items() if k != 'audit'} for source in MERGE_MANIFEST['sources']]
REPORT['duplicates_removed'] = MERGE_MANIFEST['duplicates_removed']
(WORK/'dataset_audit.json').write_text(json.dumps(REPORT, indent=2))
print('Class mapping gabungan:', NAMES)
print('Dataset:', DATASET, '| SHA256 gabungan:', DATASET_SHA256)
for source in MERGE_MANIFEST['sources']:
    print(source['source_id'], source['zip_name'], 'ID remap:', source['id_remap'])
print(json.dumps(REPORT, indent=2))


In [ ]:
# Satu contoh per kelas DARI SETIAP SUMBER; periksa remapping RED/YELLOW secara visual.
import matplotlib.pyplot as plt
palette = {'RED': (255, 0, 0), 'GREEN': (0, 180, 0), 'YELLOW': (220, 180, 0)}
fig, axes = plt.subplots(len(MERGE_MANIFEST['sources']), 3,
                         figsize=(15, 5*len(MERGE_MANIFEST['sources'])), squeeze=False)
for row_index, source in enumerate(MERGE_MANIFEST['sources']):
    candidates = [record for record in MERGE_MANIFEST['records'] if record['split'] == 'valid'
                  and any(o['source_id'] == source['source_id'] for o in record['origins'])]
    for class_id, ax in enumerate(axes[row_index]):
        ax.axis('off')
        for record in candidates:
            image_path = DATASET/'valid/images'/record['image']
            label = DATASET/'valid/labels'/(image_path.stem+'.txt')
            rows = [list(map(float, line.split())) for line in label.read_text().splitlines() if line.strip()]
            if not any(int(values[0]) == class_id for values in rows):
                continue
            frame = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
            h, w = frame.shape[:2]
            for values in rows:
                polygon = (np.array(values[1:]).reshape(-1, 2)*[w, h]).astype(np.int32)
                cv2.polylines(frame, [polygon], True, palette[NAMES[int(values[0])]], 2)
            ax.imshow(frame)
            ax.set_title(source['source_id']+' / '+NAMES[class_id])
            break
plt.tight_layout()
plt.show()


## 3. Training yang dapat dilacak dan diulang

`train` memulai dari model pretrained; `finetune` memakai `INITIAL_WEIGHTS` best.pt lama
pada dataset baru; `evaluate` mengevaluasi best.pt tanpa training ulang.
ID/nama kelas bobot lama harus sama untuk finetune/evaluate. Jika urutan dataset berubah,
gunakan mode train atau remap anotasi secara eksplisit terlebih dahulu.

AdamW, cosine learning rate, dan warmup dibuat eksplisit; finetune memakai learning rate
lebih kecil. Epoch maksimum 150 dan patience 30 bukan jaminan peningkatan; bandingkan
hasil pada test yang sama dan rekaman meja ICAM. Pertahankan nano/512 sampai latency ICAM terukur.
Hue, saturasi, BGR swap, mixup dan copy-paste dimatikan agar label warna tetap bermakna.
Augmentasi geometri/brightness ringan; frame meja nyata tetap diperlukan.

`RESUME_CHECKPOINT` khusus last.pt dari run terputus yang dibuat notebook ini, dengan ZIP dan
mapping kelas sama. Resume memulihkan argumen run lama; perubahan config baru tidak diterapkan.
Finetune menggunakan best.pt dari run selesai. Artefak/report disimpan ke Drive jika mount aktif;
dataset tetap dibaca dari disk Colab. Satu folder report unik untuk setiap sesi evaluasi.


Training menggunakan seluruh train gabungan satu kali per epoch (tanpa menggandakan dataset baru).
Crop lama lebih banyak; nilai gabungan harus dibaca bersama onnx_quality.json/by_source.
Default confidence dipilih hanya dari valid sumber terakhir (dataset matras baru, 27 gambar),
bukan dari test. Sampel ini masih kecil, sehingga uji rekaman meja tetap diperlukan.
Ubah CONFIDENCE_SOURCE='all' bila ingin valid gabungan. Resume run lama satu ZIP tidak
dapat dipakai untuk gabungan; gunakan train baru atau finetune best.pt dengan mapping sama.


In [ ]:
from datetime import datetime, timezone
import uuid

if RUN_MODE not in {'train', 'finetune', 'evaluate'}:
    raise ValueError('RUN_MODE tidak dikenal')
if RUN_MODE != 'train' and RESUME_CHECKPOINT is not None:
    raise ValueError('Resume hanya untuk mode train, bukan finetune/evaluate')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:6]
EVAL_DIR = PERSIST/'evaluation'/RUN_ID
EVAL_DIR.mkdir(parents=True, exist_ok=False)
RUN_CONTEXT = dict(dataset_sha256=DATASET_SHA256, names={str(k): v for k, v in NAMES.items()},
                   mode=RUN_MODE, seed=SEED, requested_model=MODEL, requested_imgsz=IMGSZ)
if RUN_MODE in {'finetune', 'evaluate'}:
    if INITIAL_WEIGHTS is None or not Path(INITIAL_WEIGHTS).is_file():
        raise FileNotFoundError('Isi INITIAL_WEIGHTS dengan best.pt Anda')
    model = YOLO(str(INITIAL_WEIGHTS))
    if model.task != 'segment' or model.names != NAMES:
        raise ValueError('Task/mapping kelas bobot lama berbeda dari dataset; gunakan train atau remap label')
else:
    model = YOLO(str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT is not None else MODEL)
if model.task != 'segment':
    raise ValueError('Gunakan model -seg.pt untuk anotasi poligon')

if RUN_MODE == 'evaluate':
    BEST_PT = Path(INITIAL_WEIGHTS)
    RUN_DIR = BEST_PT.parent.parent
    TRAIN_ARGS = dict(mode='evaluate', checkpoint=str(BEST_PT), imgsz=IMGSZ)
elif RESUME_CHECKPOINT is not None:
    context_path = Path(RESUME_CHECKPOINT).parent.parent/'run_context.json'
    previous = json.loads(context_path.read_text())
    if any(previous[k] != RUN_CONTEXT[k] for k in ('dataset_sha256', 'names')):
        raise ValueError('Resume memerlukan dataset ZIP dan class mapping yang sama')
    train_result = model.train(resume=True, device=DEVICE)
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_PT = RUN_DIR/'weights/best.pt'
    TRAIN_ARGS = vars(model.trainer.args).copy()
    IMGSZ = int(TRAIN_ARGS['imgsz'])
else:
    TRAIN_ARGS = dict(
        data=str(DATA_YAML), task='segment', imgsz=IMGSZ, epochs=EPOCHS,
        batch=BATCH, device=DEVICE, workers=WORKERS, seed=SEED, deterministic=True,
        project=str(RUNS_ROOT), name='arm_colors_'+RUN_ID, exist_ok=False,
        optimizer='AdamW', lr0=0.0002 if RUN_MODE == 'finetune' else 0.001,
        lrf=0.01, cos_lr=True, warmup_epochs=3.0, weight_decay=0.0005,
        save=True, save_period=10, patience=PATIENCE, cache=False, amp=(DEVICE != 'cpu'),
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.10, bgr=0.0,
        degrees=10, translate=0.05, scale=0.15, shear=0.0, perspective=0.0,
        fliplr=0.5, flipud=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0, plots=True)
    def save_run_context(trainer):
        (Path(trainer.save_dir)/'run_context.json').write_text(json.dumps(RUN_CONTEXT, indent=2))
    model.add_callback('on_pretrain_routine_start', save_run_context)
    train_result = model.train(**TRAIN_ARGS)
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_PT = RUN_DIR/'weights/best.pt'
if not BEST_PT.is_file():
    raise FileNotFoundError('best.pt tidak ditemukan; periksa training')
(EVAL_DIR/'run_context.json').write_text(json.dumps(RUN_CONTEXT, indent=2))
(EVAL_DIR/'training_args.json').write_text(json.dumps(TRAIN_ARGS, indent=2, default=str))
(EVAL_DIR/'dataset_audit.json').write_text(json.dumps(REPORT, indent=2))
shutil.copy2(DATASET/'merge_manifest.json', EVAL_DIR/'merge_manifest.json')
print('Checkpoint:', BEST_PT, '| Report:', EVAL_DIR)


In [ ]:
best = YOLO(str(BEST_PT))
assert best.names == NAMES
VALIDATION = {}
for split in ('val', 'test'):
    metrics = best.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ,
                       batch=BATCH, device=DEVICE, workers=WORKERS, plots=True,
                       project=str(EVAL_DIR), name=split, exist_ok=False)
    VALIDATION[split] = dict(box_map50=float(metrics.box.map50), box_map50_95=float(metrics.box.map),
                             mask_map50=float(metrics.seg.map50), mask_map50_95=float(metrics.seg.map),
                             per_class_mask_map50_95={NAMES[i]: float(v) for i, v in enumerate(metrics.seg.maps)})
print(json.dumps(VALIDATION, indent=2))
for filename in ('results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'MaskPR_curve.png'):
    path = RUN_DIR/filename
    if path.exists():
        display(Image(filename=str(path)))
(EVAL_DIR/'pt_metrics.json').write_text(json.dumps(VALIDATION, indent=2))


## 4. Ekspor ONNX FP32 statis

Ekspor `batch=1`, `imgsz=512`, `nms=False`, `opset=13`; runtime melakukan NMS per kelas
dan rekonstruksi mask. Input runtime memakai **stretch resize** agar sesuai preprocessing ZIP;
koordinat dikembalikan ke resolusi frame asli dengan skala X/Y terpisah.
Jika dataset baru memakai letterbox, ubah preprocessing dan pemetaan balik runtime bersama-sama.
Ultralytics menilai PT dengan preprocessing bawaannya. Laporan `onnx_quality.json`
mengukur preprocessing stretch runtime secara langsung. Jika frame dataset baru tidak persegi,
periksa perbedaan hasil PT dan ONNX ini sebelum menggunakan model di ICAM.


In [ ]:
import onnx
import onnxruntime as ort

# Copy agar export tidak menimpa ONNX run lama ketika memakai mode evaluate/finetune.
EXPORT_PT = EVAL_DIR/'best.pt'
shutil.copy2(BEST_PT, EXPORT_PT)
ONNX_PATH = Path(YOLO(str(EXPORT_PT)).export(format='onnx', imgsz=IMGSZ, batch=1, dynamic=False,
                             half=False, simplify=False, nms=False, opset=13, device='cpu'))
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
options = ort.SessionOptions()
options.intra_op_num_threads = 2
session = ort.InferenceSession(str(ONNX_PATH), sess_options=options, providers=['CPUExecutionProvider'])
print('Input:', [(t.name, t.shape, t.type) for t in session.get_inputs()])
print('Output:', [(t.name, t.shape) for t in session.get_outputs()])

# Periksa kedua output (prediksi DAN proto mask), bukan satu gambar/output saja.
paths = sorted(p for p in (DATASET/'valid/images').iterdir()
               if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
indices = np.linspace(0, len(paths)-1, min(12, len(paths)), dtype=int)
pt = YOLO(str(EXPORT_PT)).model.cpu().eval()
def tensors_in(value):
    if isinstance(value, torch.Tensor):
        yield value.detach().cpu().numpy()
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from tensors_in(item)
EXPORT_CHECK = []
for index in indices:
    frame = cv2.imread(str(paths[index]))
    rgb = cv2.cvtColor(cv2.resize(frame, (IMGSZ, IMGSZ)), cv2.COLOR_BGR2RGB)
    tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32)/255
    with torch.no_grad():
        pt_arrays = list(tensors_in(pt(torch.from_numpy(tensor))))
    outputs = session.run(None, {session.get_inputs()[0].name: tensor})
    differences = []
    for exported in outputs:
        candidates = [v for v in pt_arrays if v.shape == exported.shape]
        if len(candidates) != 1 or not np.isfinite(exported).all():
            raise ValueError('Output PT/ONNX ambigu, tidak finite, atau layout berubah')
        np.testing.assert_allclose(candidates[0], exported, rtol=1e-3, atol=1e-3)
        differences.append(float(np.max(np.abs(candidates[0]-exported))))
    EXPORT_CHECK.append(dict(image=paths[index].name, max_abs_error_per_output=differences))
(EVAL_DIR/'export_parity.json').write_text(json.dumps(EXPORT_CHECK, indent=2))
sample_path = paths[0]
sample = cv2.imread(str(sample_path))
print('Parity PT/ONNX lulus untuk', len(EXPORT_CHECK), 'gambar dan kedua output raw.')
del pt, session


## 5. Runtime ICAM dan evaluasi decoder sesungguhnya

Runtime CPU memakai preprocessing stretch/RGB/FP32 yang sama dengan model yang telah Anda uji.
Cell ini membuat script mandiri; Colab tidak membutuhkan CAMNavi2 untuk mengevaluasi gambar.
Confidence bundle otomatis dibaca dari `runtime_config.json` dan diperiksa terhadap SHA256 model.
`--threads 2` membatasi thread CPU, dapat diubah setelah benchmark di ICAM.
Decoder membatasi 20 deteksi/frame; evaluasi di bawah memakai batas yang sama.


In [ ]:
%%writefile icam_color_runtime.py
"""YOLO11-seg ONNX CPU / CAMNavi2. Detection and mapping previews only."""
import argparse
import ast
import json
import hashlib
import queue
import re
import time
from pathlib import Path

import cv2
import numpy as np

COLORS = {"RED": (0, 0, 255), "GREEN": (0, 200, 0), "YELLOW": (0, 255, 255)}


def decode(prediction, prototypes, frame_shape, input_hw, names, conf=0.6, iou=0.45):
    """Decode raw YOLO11-seg, batch=1, nms=False; input image uses stretch resize."""
    if not 0 < conf <= 1 or not 0 < iou <= 1:
        raise ValueError("conf/iou harus 0 < nilai <= 1")
    if prediction.ndim != 3 or prototypes.ndim != 4 or prediction.shape[0] != 1 or prototypes.shape[0] != 1:
        raise ValueError("Butuh output YOLO11-seg raw, batch=1")
    nc, nm = len(names), prototypes.shape[1]
    if prediction.shape[1] != 4 + nc + nm:
        raise ValueError("Layout output harus [1, 4+nc+nm, anchors]; ekspor nms=False")
    if not np.isfinite(prototypes).all():
        raise ValueError('Output proto mask tidak finite')
    rows = prediction[0].T
    rows = rows[np.isfinite(rows).all(axis=1)]
    cls = rows[:, 4:4+nc].argmax(axis=1)
    scores = rows[np.arange(len(rows)), 4+cls]
    keep = (scores >= conf) & (rows[:, 2] > 0) & (rows[:, 3] > 0)
    rows, cls, scores = rows[keep], cls[keep], scores[keep]
    boxes = np.column_stack((rows[:, :2] - rows[:, 2:4]/2, rows[:, 2:4]))
    selected = []
    for class_id in np.unique(cls):
        indices = np.flatnonzero(cls == class_id)
        local = cv2.dnn.NMSBoxes(boxes[indices].tolist(), scores[indices].tolist(), conf, iou)
        selected.extend(indices[np.asarray(local, dtype=int).reshape(-1)].tolist())
    h, w = frame_shape[:2]
    ih, iw = input_hw
    proto = prototypes[0]
    detections = []
    for index in sorted(selected, key=lambda k: -scores[k])[:20]:
        x, y, bw, bh = boxes[index]
        x1, x2 = np.clip([x*w/iw, (x+bw)*w/iw], 0, w).astype(int)
        y1, y2 = np.clip([y*h/ih, (y+bh)*h/ih], 0, h).astype(int)
        if x2 <= x1 or y2 <= y1:
            continue
        logits = (rows[index, 4+nc:] @ proto.reshape(nm, -1)).reshape(proto.shape[1:])
        mask = (cv2.resize(logits, (w, h))[y1:y2, x1:x2] > 0).astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        center = [x1 + moments['m10']/moments['m00'], y1 + moments['m01']/moments['m00']]
        # An irregular mask centroid can lie outside the object: do not offer it for motion.
        center_inside = cv2.pointPolygonTest(contour, (center[0]-x1, center[1]-y1), False) >= 0
        detections.append(dict(class_id=int(cls[index]), color=names[int(cls[index])],
                               confidence=float(scores[index]), center_px=center,
                               center_inside_mask=center_inside, bbox_xyxy=[int(x1), int(y1), int(x2), int(y2)],
                               contour=(contour.reshape(-1, 2)+[x1, y1]).tolist()))
    return detections


class ColorModel:
    def __init__(self, model_path, threads=2):
        import onnxruntime as ort
        if not isinstance(threads, int) or threads < 1:
            raise ValueError('threads harus integer positif')
        options = ort.SessionOptions()
        options.intra_op_num_threads = threads
        self.session = ort.InferenceSession(str(model_path), sess_options=options, providers=['CPUExecutionProvider'])
        self.default_conf = 0.60
        config_path = Path(model_path).with_name('runtime_config.json')
        if config_path.is_file():
            config = json.loads(config_path.read_text())
            digest = hashlib.sha256(Path(model_path).read_bytes()).hexdigest()
            if config['model_sha256'] != digest or config['preprocessing'] != 'stretch_rgb_nchw_fp32':
                raise ValueError('runtime_config tidak cocok dengan model/preprocessing')
            self.default_conf = float(config['confidence'])
            if not 0 < self.default_conf <= 1:
                raise ValueError('Confidence runtime tidak valid')
        tensor = self.session.get_inputs()[0]
        if tensor.type != 'tensor(float)' or len(tensor.shape) != 4 or tensor.shape[:2] != [1, 3]:
            raise ValueError('Ekspor ONNX FP32 NCHW batch=1 diperlukan')
        if not all(isinstance(v, int) and v > 0 for v in tensor.shape):
            raise ValueError('Ekspor dynamic=False diperlukan')
        self.input_name, self.hw = tensor.name, tuple(tensor.shape[2:])
        meta = self.session.get_modelmeta().custom_metadata_map
        names = ast.literal_eval(meta.get('names', '{}'))
        self.names = {int(k): str(v) for k, v in names.items()}
        if sorted(self.names) != [0, 1, 2] or set(self.names.values()) != set(COLORS):
            raise ValueError('Metadata names harus tepat GREEN, YELLOW, RED dengan ID 0..2')

    def predict(self, frame, conf=None):
        conf = self.default_conf if conf is None else conf
        if frame is None or frame.ndim != 3 or frame.shape[2] != 3:
            raise ValueError('Frame harus BGR 3 channel; kamera mono tidak dapat mendeteksi warna')
        rgb = cv2.cvtColor(cv2.resize(frame, (self.hw[1], self.hw[0])), cv2.COLOR_BGR2RGB)
        tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32)/255.0
        outputs = self.session.run(None, {self.input_name: tensor})
        predictions = [v for v in outputs if v.ndim == 3]
        prototypes = [v for v in outputs if v.ndim == 4]
        if len(predictions) != 1 or len(prototypes) != 1:
            raise ValueError('Model harus YOLO11 instance segmentation dengan dua output raw')
        return decode(predictions[0], prototypes[0], frame.shape, self.hw, self.names, conf)


class CamNaviSource:
    """CAMNavi2 JPEG callback, latest frame only. Requires vendor BSP SDK."""
    def __init__(self, device_name, width=640, height=480):
        from CamNavi2 import CamNavi2
        self.sdk = CamNavi2.CamNavi2() if hasattr(CamNavi2, 'CamNavi2') else CamNavi2()
        print('CAMNavi devices:', self.sdk.enum_camera_list())
        self.camera = self.sdk.get_device_by_name(device_name)
        self.frames = queue.Queue(maxsize=1)
        if self.camera is None:
            raise RuntimeError('Nama device tidak ditemukan; gunakan hasil enum_camera_list()')
        if int(self.sdk.advcam_query_fw_sku(self.camera)) != 1:
            raise RuntimeError('SDK tidak melaporkan kamera color (SKU=1); verifikasi BSP')
        try:
            self.sdk.advcam_config_pipeline(self.camera, acq_mode=0, width=width, height=height,
                                           enable_infer=0, pipeline_mode='default')
            self.sdk.advcam_open(self.camera)
            self.sdk.advcam_register_new_image_handler(self.camera, self._sample)
            self.sdk.advcam_play(self.camera)
        except BaseException:
            self.close()
            raise

    def _sample(self, sample):
        if sample is None:
            return
        received_at = time.monotonic()  # Stamp callback entry before JPEG decoding.
        buf = sample.get_buffer()
        frame = cv2.imdecode(np.frombuffer(buf.extract_dup(0, buf.get_size()), np.uint8), cv2.IMREAD_UNCHANGED)
        if frame is None or frame.ndim != 3 or frame.shape[2] != 3:
            return
        if self.frames.full():
            try:
                self.frames.get_nowait()
            except queue.Empty:
                pass
        try:
            self.frames.put_nowait((received_at, frame))
        except queue.Full:
            pass

    def read_sample(self, timeout=5, after=None):
        deadline = time.monotonic()+timeout
        while time.monotonic() < deadline:
            try:
                stamp, frame = self.frames.get(timeout=max(0.001, deadline-time.monotonic()))
            except queue.Empty:
                break
            if time.monotonic()-stamp < 1 and (after is None or stamp > after):
                return stamp, frame
        raise TimeoutError('Tidak ada frame JPEG berwarna baru dari CAMNavi2')

    def read(self, timeout=5):
        return self.read_sample(timeout=timeout)[1]

    def close(self):
        try:
            self.sdk.advcam_register_new_image_handler(self.camera, None)
        finally:
            self.sdk.advcam_close(self.camera)


def annotate(frame, detections):
    out = frame.copy()
    for d in detections:
        color = COLORS[d['color']]
        cv2.polylines(out, [np.asarray(d['contour'], np.int32)], True, color, 2)
        p = tuple(np.round(d['center_px']).astype(int))
        cv2.circle(out, p, 4, color, -1)
        cv2.putText(out, f"{d['color']} {d['confidence']:.2f}", p,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return out


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--model', default='best.onnx')
    parser.add_argument('--source', default='camnavi', help='camnavi, image path, video, RTSP, or webcam index')
    parser.add_argument('--device-name', default='iCam500', help='Use actual CAMNavi enum name; manual uses iCam500')
    parser.add_argument('--width', type=int, default=640)
    parser.add_argument('--height', type=int, default=480)
    parser.add_argument('--conf', type=float, default=None, help='Override runtime_config confidence')
    parser.add_argument('--threads', type=int, default=2)
    parser.add_argument('--calibration', help='Optional robot_plane.json; output XY coordinates only')
    parser.add_argument('--show', action='store_true', help='Requires GUI OpenCV and local display')
    parser.add_argument('--frames', type=int, default=100)
    args = parser.parse_args()
    model = ColorModel(args.model, threads=args.threads)
    calibration = None
    if args.calibration:
        from robot_mapping import RobotPlane
        calibration = RobotPlane(json.loads(Path(args.calibration).read_text()))
    source = None
    try:
        still = cv2.imread(args.source) if Path(args.source).suffix.lower() in {'.jpg', '.jpeg', '.png'} else None
        if args.source == 'camnavi':
            source = CamNaviSource(args.device_name, args.width, args.height)
        elif still is None:
            source = cv2.VideoCapture(int(args.source) if args.source.isdigit() else args.source)
            if not source.isOpened():
                raise RuntimeError('Sumber kamera/video tidak dapat dibuka')
        for _ in range(args.frames):
            start = time.monotonic()
            if still is not None:
                frame = still
            elif isinstance(source, CamNaviSource):
                frame = source.read()
            else:
                ok, frame = source.read()
                if not ok:
                    break
            detections = model.predict(frame, args.conf)
            for detection in detections:
                detection['inside_work_area'] = None  # Unknown until mat calibration is supplied.
                if calibration and detection['center_inside_mask']:
                    xy = calibration.xy(detection['center_px'], frame.shape)
                    detection['inside_work_area'] = xy is not None
                    detection['xy_mm'] = xy
                    detection['plane_z_mm'] = calibration.z  # Not measured depth, no motion command.
            print(json.dumps(dict(detections=detections, elapsed_ms=round((time.monotonic()-start)*1000, 1))))
            overlay = annotate(frame, detections)
            if still is not None:
                cv2.imwrite('prediction.jpg', overlay)
                break
            if args.show:
                cv2.imshow('RED GREEN YELLOW', overlay)
                if cv2.waitKey(1) & 0xff in (27, ord('q')):
                    break
    finally:
        if isinstance(source, CamNaviSource):
            source.close()
        elif source is not None:
            source.release()
        if args.show:
            cv2.destroyAllWindows()


if __name__ == '__main__':
    main()


In [ ]:
%%writefile arm_quality.py
"""Evaluation of the actual deployed decoder; no training or robot motion."""
from pathlib import Path
import time
import cv2
import numpy as np


def image_paths(directory):
    return sorted(p for p in Path(directory).iterdir()
                  if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})


def ground_truth(label_path, shape):
    h, w = shape[:2]
    objects = []
    for line in Path(label_path).read_text().splitlines():
        if not line.strip():
            continue
        values = np.asarray([float(v) for v in line.split()])
        points = values[1:].reshape(-1, 2) * [w, h]
        moment = cv2.moments(points.astype(np.float32))
        objects.append(dict(class_id=int(values[0]), contour=points.tolist(),
                            bbox_xyxy=[*points.min(0), *points.max(0)],
                            center_px=[moment['m10']/moment['m00'], moment['m01']/moment['m00']]))
    return objects


def box_iou(a, b):
    a, b = np.asarray(a), np.asarray(b)
    intersection = np.maximum(0, np.minimum(a[2:], b[2:])-np.maximum(a[:2], b[:2])).prod()
    union = np.maximum(0, a[2:]-a[:2]).prod() + np.maximum(0, b[2:]-b[:2]).prod() - intersection
    return float(intersection / union) if union > 0 else 0.0


def match_objects(predictions, truths, threshold, iou=0.5):
    """Confidence-ordered, class-aware, one-to-one matching (not mAP)."""
    used, matches, false_positives = set(), [], []
    for prediction in sorted(predictions, key=lambda d: -d['confidence']):
        if prediction['confidence'] < threshold:
            continue
        candidates = [(box_iou(prediction['bbox_xyxy'], truth['bbox_xyxy']), index)
                      for index, truth in enumerate(truths)
                      if index not in used and truth['class_id'] == prediction['class_id']]
        overlap, index = max(candidates, default=(0, -1))
        if index >= 0 and overlap >= iou:
            used.add(index)
            matches.append((prediction, truths[index]))
        else:
            false_positives.append(prediction)
    return matches, false_positives, [t for i, t in enumerate(truths) if i not in used]


def summarize(records, names, threshold, localization=False):
    counts = {name: dict(tp=0, fp=0, fn=0) for name in names.values()}
    errors, mask_ious, cases = [], [], []
    negative_images = negative_false_positives = 0
    for record in records:
        matches, fps, fns = match_objects(record['predictions'], record['truths'], threshold)
        for prediction, truth in matches:
            counts[names[truth['class_id']]]['tp'] += 1
            if localization:
                errors.append(float(np.linalg.norm(np.asarray(prediction['center_px'])-truth['center_px'])))
                masks = []
                for item in (prediction, truth):
                    mask = np.zeros(record['shape'][:2], dtype=np.uint8)
                    cv2.fillPoly(mask, [np.round(item['contour']).astype(np.int32)], 1)
                    masks.append(mask.astype(bool))
                union = np.logical_or(*masks).sum()
                mask_ious.append(float(np.logical_and(*masks).sum()/union) if union else 0.)
        for p in fps:
            counts[names[p['class_id']]]['fp'] += 1
        for t in fns:
            counts[names[t['class_id']]]['fn'] += 1
        if not record['truths']:
            negative_images += 1
            negative_false_positives += bool(fps)
        if fps or fns:
            cases.append(dict(image=record['image'], fp=len(fps), fn=len(fns)))
    total = {key: sum(c[key] for c in counts.values()) for key in ('tp', 'fp', 'fn')}
    def rates(count):
        tp, fp, fn = (count[k] for k in ('tp', 'fp', 'fn'))
        return dict(count, precision=tp/(tp+fp) if tp+fp else 0.,
                    recall=tp/(tp+fn) if tp+fn else 0.,
                    f05=1.25*tp/(1.25*tp+fp+.25*fn) if tp+fp+fn else 0.)
    return dict(threshold=float(threshold), overall=rates(total),
                per_class={k: rates(v) for k, v in counts.items()},
                negative_images=negative_images,
                negative_fp_image_rate=negative_false_positives/negative_images if negative_images else None,
                matched_centroid_error_px_p50=float(np.median(errors)) if errors else None,
                matched_centroid_error_px_p95=float(np.percentile(errors, 95)) if errors else None,
                matched_mask_iou_mean=float(np.mean(mask_ious)) if mask_ious else None,
                error_cases=sorted(cases, key=lambda c: -(c['fp']+c['fn'])))


def select_threshold(records, names, default=0.60, min_precision=0.98):
    grid = sorted(set([0.25, 0.35, 0.45, 0.55, float(default), 0.65, 0.75, 0.85, 0.95]))
    scores = [summarize(records, names, value) for value in grid]
    eligible = [s for s in scores if all(c['precision'] >= min_precision and c['tp'] > 0
                                       for c in s['per_class'].values())]
    chosen = max(eligible, key=lambda s: (s['overall']['f05'], -abs(s['threshold']-default))) if eligible else None
    return dict(selected=chosen['threshold'] if chosen else default,
                precision_target_met=bool(chosen), min_precision=min_precision,
                selection_split='valid only; test never used for selection', candidates=scores)


def collect_predictions(detector, base, split):
    paths = image_paths(Path(base)/split/'images')
    # Warm up separately; CPU timings here describe the Colab/PC host, not ICAM.
    for _ in range(3):
        detector.predict(cv2.imread(str(paths[0])), conf=0.25)
    records, elapsed = [], []
    for path in paths:
        frame = cv2.imread(str(path))
        started = time.perf_counter()
        predictions = detector.predict(frame, conf=0.25)
        elapsed.append((time.perf_counter()-started)*1000)
        records.append(dict(image=str(path), shape=frame.shape, predictions=predictions,
                            truths=ground_truth(Path(base)/split/'labels'/(path.stem+'.txt'), frame.shape)))
    return records, dict(host='Colab/PC CPU, NOT ICAM-300', images=len(records),
                         inference_decode_ms_p50=float(np.median(elapsed)),
                         inference_decode_ms_p95=float(np.percentile(elapsed, 95)))


In [ ]:
import importlib
import icam_color_runtime
import arm_quality
importlib.reload(icam_color_runtime)  # Aman: modul Python kita sendiri, bukan NumPy/extension biner.
importlib.reload(arm_quality)
from icam_color_runtime import ColorModel, annotate
from arm_quality import collect_predictions, select_threshold, summarize

detector = ColorModel(ONNX_PATH)
valid_records, valid_timing = collect_predictions(detector, DATASET, 'valid')
if CONFIDENCE_SOURCE not in {'last', 'all'}:
    raise ValueError("CONFIDENCE_SOURCE harus 'last' atau 'all'")
confidence_source = MERGE_MANIFEST['sources'][-1]
calibration_records = (records_for_source(valid_records, MERGE_MANIFEST['records'], confidence_source['source_id'])
                       if CONFIDENCE_SOURCE == 'last' else valid_records)
if not calibration_records:
    raise ValueError('Tidak ada validation untuk sumber confidence yang dipilih')
THRESHOLD_REPORT = select_threshold(calibration_records, NAMES, default=CONF, min_precision=MIN_CLASS_PRECISION)
THRESHOLD_REPORT['source'] = confidence_source['zip_name'] if CONFIDENCE_SOURCE == 'last' else 'all merged sources'
THRESHOLD_REPORT['validation_images'] = len(calibration_records)
if TUNE_CONFIDENCE:
    CONF = THRESHOLD_REPORT['selected']
THRESHOLD_REPORT['applied_confidence'] = CONF
THRESHOLD_REPORT['tuning_enabled'] = TUNE_CONFIDENCE
THRESHOLD_REPORT['coverage'] = REPORT['coverage']
if not REPORT['coverage']['all_splits_have_negatives']:
    print('Belum ada negatif pada semua split; false alarm di meja kosong belum terukur.')
if not THRESHOLD_REPORT['precision_target_met']:
    print('Target precision per kelas belum tercapai; threshold default dipertahankan.')
test_records, test_timing = collect_predictions(detector, DATASET, 'test')
ONNX_QUALITY = dict(
    definition='Operating point: class + bbox IoU>=0.5, greedy one-to-one; bukan mAP',
    valid=summarize(valid_records, NAMES, CONF, localization=True),
    test=summarize(test_records, NAMES, CONF, localization=True),
    latency=dict(valid=valid_timing, test=test_timing),
    localization_note='Centroid error hanya pasangan TP, dalam piksel; bukan akurasi genggam/mm')
ONNX_QUALITY['by_source'] = {}
for source in MERGE_MANIFEST['sources']:
    source_metrics = dict(zip_name=source['zip_name'])
    for split, records in [('valid', valid_records), ('test', test_records)]:
        selected_records = records_for_source(records, MERGE_MANIFEST['records'], source['source_id'])
        source_metrics[split] = dict(images=len(selected_records),
                                    **summarize(selected_records, NAMES, CONF, localization=True))
    ONNX_QUALITY['by_source'][source['source_id']] = source_metrics
    print(source['zip_name'], '| ONNX test:', source_metrics['test']['overall'])
(EVAL_DIR/'confidence_selection.json').write_text(json.dumps(THRESHOLD_REPORT, indent=2))
(EVAL_DIR/'onnx_quality.json').write_text(json.dumps(ONNX_QUALITY, indent=2))
RUNTIME_CONFIG = dict(model_sha256=hashlib.sha256(ONNX_PATH.read_bytes()).hexdigest(),
                      preprocessing='stretch_rgb_nchw_fp32', confidence=CONF, names=NAMES)
(ONNX_PATH.parent/'runtime_config.json').write_text(json.dumps(RUNTIME_CONFIG, indent=2))
print('Confidence:', CONF, '| ONNX test:', ONNX_QUALITY['test']['overall'])
print('Per warna:', ONNX_QUALITY['test']['per_class'])
print('Centroid p95 (px):', ONNX_QUALITY['test']['matched_centroid_error_px_p95'])
print('Latency host CPU:', test_timing)

# Simpan sampai 12 contoh FP/FN untuk ditinjau, label GT putih, prediksi berwarna.
error_dir = EVAL_DIR/'error_examples'
error_dir.mkdir(exist_ok=True)
by_path = {r['image']: r for r in test_records}
for rank, case in enumerate(ONNX_QUALITY['test']['error_cases'][:12]):
    record = by_path[case['image']]
    overlay = annotate(cv2.imread(record['image']), [p for p in record['predictions'] if p['confidence'] >= CONF])
    for truth in record['truths']:
        cv2.polylines(overlay, [np.round(truth['contour']).astype(np.int32)], True, (255,255,255), 1)
    cv2.imwrite(str(error_dir/f'{rank:02d}_{Path(record["image"]).stem}.jpg'), overlay)


In [ ]:
from icam_color_runtime import ColorModel, annotate

detector = ColorModel(ONNX_PATH)
detections = detector.predict(sample, conf=CONF)
display(Image(data=cv2.imencode('.jpg', annotate(sample, detections))[1].tobytes()))
print([{k: v for k, v in d.items() if k != 'contour'} for d in detections])

# Isi dengan foto frame PENUH meja ICAM-300 untuk uji multiobjek yang bermakna.
SCENE_IMAGE = None  # Contoh: ROOT/'meja_3_warna.jpg'
if SCENE_IMAGE is not None:
    scene = cv2.imread(str(SCENE_IMAGE))
    if scene is None:
        raise ValueError('SCENE_IMAGE tidak dapat dibaca')
    found = detector.predict(scene, CONF)
    display(Image(data=cv2.imencode('.jpg', annotate(scene, found))[1].tobytes()))
    print(Counter(d['color'] for d in found))


## 6. Mapping untuk SmoothingPitch.ino (pratinjau)

Titik warna berikut persis dari tabel Anda. Belum ditetapkan sebagai pick atau drop.
Model memberi **warna dan piksel**; tabel memberi **pose referensi robot**. Keduanya berbeda:
warna RED tidak membuktikan bahwa benda berada di X=10,Y=550.

Setelah dudukan ICAM tetap, ukur empat pasangan piksel ↔ XY robot pada bidang Z yang sama,
berurutan mengelilingi bidang. Uji beberapa titik tambahan yang tidak dipakai membuat homography
dan catat error mm. `plane_z_mm` adalah tinggi yang diukur, bukan prediksi kedalaman kamera.
Kalibrasi harus diulang jika kamera, crop/resolusi, lensa, atau tinggi bidang berubah.
Pusat mask hanya kandidat lokasi genggam.

Temuan firmware yang memengaruhi tahap mapping:

- `HOME` bergerak ke pose tegak; `ZERO` hanya menetapkan nol saat pose fisik sudah sesuai.
- PTP bergerak sinkron dalam ruang joint, bukan garis lurus XYZ.
- Parabolik menghitung 40 waypoint. Firmware memeriksa waypoint sambil bergerak; jika gagal
  di tengah, sebagian gerak sudah terjadi dan `curX/Y/Z` masih nilai awal. Preview di bawah
  memeriksa seluruh waypoint dahulu, tetapi tidak menggantikan uji mekanis.
- `runSpeedToPosition()` memblokir loop; firmware ini tidak punya handler `STOP`.
- Servo ditulis setelah gerakan selesai; angka servo 0 belum berarti membuka/menutup gripper.
- IK tidak memeriksa seluruh batas joint, kabel, meja, dan tabrakan antarlink.

Notebook tidak membuka serial. Perintah yang tercetak adalah bahan pemeriksaan mapping,
bukan urutan pick-and-place atau bukti lintasan bebas tabrakan.


**Bidang kerja = matras hijau.** Isi empat points_px dari sudut fisik matras pada
frame ICAM asli (bukan gambar training 512 yang sudah diaugmentasi), lalu pasangan
points_robot_xy_mm pada sudut yang sama. Kelas model tetap tiga warna objek;
matras tidak ditambahkan sebagai objek GREEN. Warna hijau background tidak dibuang dengan HSV.
Runtime memberi inside_work_area berdasarkan pusat mask dan poligon matras terkalibrasi.
Pusat di luar matras tidak mendapat XY; kalibrasi kosong berarti status area belum diketahui.
Keanggotaan pusat tidak menjamin seluruh benda berada di dalam matras atau layak digenggam.


In [ ]:
%%writefile robot_mapping.py
"""SmoothingPitch.ino mapping previews only. Does not open serial or move a robot."""
import math
import numpy as np
import cv2

L1, L2, L3, L_TOOL, R_BASE = 231.5, 221.124, 223.22, 175.0, 80.0
HOME_POSE = [0., 0., L1+L2+L3+L_TOOL, 0., 0., 0.]


def firmware_ik(pose):
    """Mirror firmware workspace/IK checks, not a collision or joint-limit proof."""
    values = np.asarray(pose, dtype=float)
    if values.shape != (6,) or not np.isfinite(values).all():
        raise ValueError('Pose harus enam angka finite: X Y Z Pitch DOF4 Servo')
    x, y, z, pitch, dof4, servo = values
    if not (0 <= x <= 600 and -600 <= y <= 600 and 0 <= z <= HOME_POSE[2] and 0 <= servo <= 180):
        raise ValueError('Di luar workspace/servo firmware')
    radius = math.hypot(x, y)
    if z <= L1 and radius < R_BASE:
        raise ValueError('Target menabrak base robot')
    pitch_rad = math.radians(pitch)
    rw, zw = radius-L_TOOL*math.sin(pitch_rad), z-L1-L_TOOL*math.cos(pitch_rad)
    distance = math.hypot(rw, zw)
    if distance > L2+L3+.01 or distance < abs(L2-L3)-.01:
        raise ValueError('Target di luar jangkauan IK')
    theta3 = math.acos(np.clip((distance**2-L2**2-L3**2)/(2*L2*L3), -1, 1))
    theta2 = math.atan2(rw, zw)-math.atan2(L3*math.sin(theta3), L2+L3*math.cos(theta3))
    r2, z2 = L2*math.sin(theta2), L1+L2*math.cos(theta2)
    r3, z3 = r2+L3*math.sin(theta2+theta3), z2+L3*math.cos(theta2+theta3)
    if min(z2, z3) < 0 or (z2 <= L1 and abs(r2) < R_BASE) or (z3 <= L1 and abs(r3) < R_BASE):
        raise ValueError('IK melanggar base/lantai menurut firmware')
    j2, j3 = math.degrees(theta2), math.degrees(theta3)
    return [math.degrees(math.atan2(y, x)) if x or y else 0., j2, j3, dof4, pitch-j2-j3, servo]


def command_preview(pose, mode='PTP', travel_h=100.):
    firmware_ik(pose)
    if mode not in {'PTP', 'PARABOLIC'}:
        raise ValueError('Mode harus PTP atau PARABOLIC')
    values = list(pose)
    if mode == 'PARABOLIC':
        if not math.isfinite(travel_h) or travel_h <= 0:
            raise ValueError('Travel H harus positif untuk mode PARABOLIC')
        values.append(travel_h)
    return ' '.join(f'{v:.3f}'.rstrip('0').rstrip('.') if v else '0' for v in values)


def check_parabolic(start_pose, target_pose, travel_h):
    """Check all 40 waypoints BEFORE sending anything (no sending here)."""
    command_preview(target_pose, 'PARABOLIC', travel_h)
    firmware_ik(start_pose)
    start, target = np.asarray(start_pose, float), np.asarray(target_pose, float)
    for i in range(1, 41):
        t = i/40
        pose = start+(target-start)*t
        pose[2] += 4*travel_h*t*(1-t)
        try:
            firmware_ik(pose)
        except ValueError as error:
            return dict(passed=False, waypoint=i, t=t, reason=str(error))
    return dict(passed=True, waypoints_checked=40,
                limitation='IK/workspace saja; belum memeriksa joint limit dan tabrakan sepanjang lintasan fisik')


class RobotPlane:
    """Four corresponding pixels and robot XY points on one fixed-height plane."""
    def __init__(self, config):
        self.size = tuple(config['frame_size_wh'])
        self.z = float(config['plane_z_mm'])
        self.pixels = np.asarray(config['points_px'], np.float32)
        self.robot = np.asarray(config['points_robot_xy_mm'], np.float32)
        if len(self.size) != 2 or min(self.size) <= 0 or not math.isfinite(self.z):
            raise ValueError('Resolusi/Z kalibrasi tidak valid')
        for points in (self.pixels, self.robot):
            if points.shape != (4, 2) or not np.isfinite(points).all() or not cv2.isContourConvex(points) or abs(cv2.contourArea(points)) < 1:
                raise ValueError('Isi 4 pasangan titik cembung berurutan pada satu bidang')
        if (self.pixels < 0).any() or (self.pixels >= self.size).any():
            raise ValueError('Piksel kalibrasi di luar frame')
        self.matrix = cv2.getPerspectiveTransform(self.pixels, self.robot)
        if not np.isfinite(self.matrix).all() or np.linalg.matrix_rank(self.matrix) < 3:
            raise ValueError('Homography degenerat')

    def xy(self, pixel, frame_shape):
        if (frame_shape[1], frame_shape[0]) != self.size:
            raise ValueError('Resolusi berubah, ulangi kalibrasi')
        point = np.asarray(pixel, np.float32)
        if point.shape != (2,) or not np.isfinite(point).all():
            raise ValueError('Piksel tidak valid')
        if cv2.pointPolygonTest(self.pixels, tuple(map(float, point)), False) < 0:
            return None
        xy = cv2.perspectiveTransform(point.reshape(1, 1, 2), self.matrix)[0, 0]
        if not np.isfinite(xy).all() or cv2.pointPolygonTest(self.robot, tuple(map(float, xy)), True) < -0.01:
            return None
        return xy.tolist()


In [ ]:
import robot_mapping
importlib.reload(robot_mapping)
from robot_mapping import RobotPlane, command_preview, check_parabolic, HOME_POSE

ROBOT_MAPPING = {
    'firmware': 'SmoothingPitch.ino', 'baud': 115200, 'units': 'mm / degrees',
    'role': 'reference_only_pending_pick_or_drop_confirmation',
    'poses': {'RED': [10, 550, 125, 90, 0, 0],
              'GREEN': [10, 410, 125, 90, 0, 0],
              'YELLOW': [153, 530, 125, 90, 0, 0],
              'HOME': [0, 0, 850.844, 0, 0, 0]},
    'travel_h_mm': 100,
}
CALIBRATION = {
    'frame_size_wh': [640, 480],  # Ganti sesuai resolusi frame ICAM sebenarnya.
    'plane_z_mm': None,  # Tinggi bidang matras yang DIUKUR, bukan otomatis Z pose warna.
    'points_px': None,  # Empat sudut matras pada frame ICAM asli, berurutan tanpa silang.
    'points_robot_xy_mm': None,  # Empat XY robot untuk titik fisik yang sama, urutan sama.
}
calibration = None
if all(CALIBRATION[k] is not None for k in ('plane_z_mm', 'points_px', 'points_robot_xy_mm')):
    calibration = RobotPlane(CALIBRATION)
    print('Homography terbentuk; verifikasi error mm pada titik independen di meja.')
else:
    print('Piksel -> XY belum aktif. Isi hasil pengukuran setelah dudukan ICAM selesai.')
for color, pose in ROBOT_MAPPING['poses'].items():
    print(color, 'PTP:', command_preview(pose))
    if color != 'HOME':
        print('  PARABOLIC:', command_preview(pose, 'PARABOLIC', ROBOT_MAPPING['travel_h_mm']))


### Preview lintasan dari pose awal yang diketahui

`START_POSE=None` karena pose aktual robot belum dibaca/diukur. Isi enam angka hanya untuk
simulasi dari pose yang sudah diketahui. Pemeriksaan mencerminkan rumus IK firmware dan
menambahkan batas workspace di setiap waypoint. Target lolos IK belum berarti seluruh
lintasan lolos; terutama gerakan dari HOME dengan Travel H=100 harus diperiksa tersendiri.


In [ ]:
START_POSE = None  # [X,Y,Z,Pitch,DOF4,Servo] aktual; jangan asumsikan HOME setelah reset.
TARGET_COLOR = 'RED'
PATH_PREVIEW = None
if START_POSE is not None:
    PATH_PREVIEW = check_parabolic(START_POSE, ROBOT_MAPPING['poses'][TARGET_COLOR],
                                   ROBOT_MAPPING['travel_h_mm'])
    print(json.dumps(PATH_PREVIEW, indent=2))

# Preview XY untuk frame SCENE_IMAGE saja, tidak untuk crop dataset yang berbeda kamera.
if calibration is not None and SCENE_IMAGE is not None:
    for detection in found:
        if detection['center_inside_mask']:
            print(detection['color'], 'pixel:', detection['center_px'],
                  'XY mm:', calibration.xy(detection['center_px'], scene.shape),
                  'Z bidang:', calibration.z)


## 6b. Satu stage = ambil satu benda → mangkok warna → HOME

ICAM memilih satu target dalam matras dari frame baru, lalu mengunci XY dan warna.
Arduino menangani seluruh urutan motor/gripper dan memilih mangkok dari flag G/R/Y.
Selama menunggu penerimaan maupun gerakan, ICAM tidak memilih atau mengantrekan target lain.
Setelah Arduino menyatakan seluruh urutan selesai di HOME, buang hasil/frame selama stage
dan pilih kembali dari frame yang diterima callback SETELAH pesan selesai diterima.

State: WAIT_READY → IDLE → WAIT_ACCEPTED → BUSY → IDLE.
ERROR, timeout, putus koneksi, atau reset ESP32 → FAULT; tidak ada retry gerakan otomatis.
READY/OK berulang saat BUSY tidak membuka pengunci. DONE dengan ID stage lain diabaikan.

**Status implementasi:** Terima_data_icamv2.ino yang diperiksa baru menerima XY + one-hot
dan membalas teks OK. Fungsi ambil/mangkok/HOME belum dipanggil di file itu.
OK berarti data diterima, bukan gerakan selesai. Modul berikut adalah pengatur stage yang
sudah dapat disimulasikan, belum koneksi TCP atau implementasi motor. Sambungan gerak nyata
menunggu firmware yang menjalankan siklus dan mengirim status selesai.

Kontrak integrasi yang disiapkan (JSON satu baris + newline, TCP port 5000):

- Arduino → ICAM: {"event":"READY","home":true}, hanya setelah kondisi awal siap/HOME terverifikasi.
- ICAM → Arduino: {"stage_id":"ID_UNIK","x":10.0,"y":550.0,"G":0,"R":1,"Y":0}.
- Arduino mengunci salinan target, lalu membalas {"event":"ACCEPTED","stage_id":"ID_UNIK"}.
- Arduino mengambil benda, menaruh ke mangkok yang ditentukan kodenya, lalu HOME.
- Hanya setelah HOME selesai: {"event":"DONE","stage_id":"ID_UNIK","home":true}.
- Bila gagal: {"event":"ERROR","stage_id":"ID_UNIK","reason":"..."}; jangan mengirim DONE.

Firmware harus menolak target selama busy, mengenali ID yang sudah diterima agar tidak
mengeksekusi duplikat, dan tidak memakai koordinat baru untuk menimpa stage aktif.
DONE wajib muncul setelah gerak HOME selesai, bukan saat perintah HOME baru dikirim.
Koordinator dipanggil dari satu event loop; adapter TCP nanti memanggil fail() saat koneksi putus.
Posisi mangkok dan urutan servo tetap milik firmware, bukan ditentukan YOLO.

Timestamp frame memakai jam monotonic ICAM yang sama dengan koordinator. CAMNavi
read_sample() mengembalikan timestamp callback dan frame; jangan memberi timestamp baru
pada gambar lama. Resolusi dan kalibrasi matras tetap harus sesuai.


In [ ]:
%%writefile stage_coordinator.py
"""One active pick/place/HOME stage. Pure coordinator: no network or motor commands."""
import copy
import math
import time
import uuid


class StageCoordinator:
    def __init__(self, min_confidence=0.6, max_frame_age=1.0, timeout=180.0, clock=time.monotonic):
        for value in (min_confidence, max_frame_age, timeout):
            if isinstance(value, bool) or not math.isfinite(value) or value <= 0:
                raise ValueError('Confidence, frame age, timeout harus positif dan finite')
        if min_confidence > 1:
            raise ValueError('Confidence maksimal 1')
        self.min_confidence = min_confidence
        self.max_frame_age = max_frame_age
        self.timeout = timeout
        self.clock = clock
        self.state = 'WAIT_READY'
        self.fresh_after = None
        self._active = None
        self._deadline = None
        self.reason = None

    @property
    def active(self):
        return copy.deepcopy(self._active)

    def fail(self, reason):
        # No automatic reconnect, replay, or retry: physical outcome may be unknown.
        self.state = 'FAULT'
        self.reason = str(reason)

    def tick(self):
        if self.state in ('WAIT_ACCEPTED', 'BUSY') and self.clock() >= self._deadline:
            self.fail('Timeout menunggu stage selesai di HOME; jangan kirim ulang otomatis')
        return self.state

    def on_status(self, message):
        """Future Arduino status contract. Legacy 'OK' never completes a stage."""
        self.tick()
        if self.state == 'FAULT' or not isinstance(message, dict):
            return False
        event = message.get('event')
        if event == 'RESET':
            self.fail('ESP32 reset; status fisik robot perlu diperiksa')
            return False
        if event == 'READY':
            if self.state != 'WAIT_READY' or message.get('home') is not True:
                return False
            self.state = 'IDLE'
            self.fresh_after = self.clock()
            return True
        if self._active is None or message.get('stage_id') != self._active['stage_id']:
            return False
        if event == 'ERROR':
            self.fail(message.get('reason', 'Firmware melaporkan stage gagal'))
            return False
        if event == 'ACCEPTED' and self.state == 'WAIT_ACCEPTED':
            self.state = 'BUSY'
            return True
        if event == 'DONE':
            if self.state != 'BUSY' or message.get('home') is not True:
                self.fail('DONE sebelum ACCEPTED atau belum kembali HOME')
                return False
            self.state = 'IDLE'
            self.fresh_after = self.clock()
            self._active = None
            self._deadline = None
            return True
        return False

    def offer(self, detections, captured_at):
        """Pick from one fresh frame only. Drop all candidates offered while busy."""
        self.tick()
        if self.state != 'IDLE':
            return None
        now = self.clock()
        if (isinstance(captured_at, bool) or not isinstance(captured_at, (int, float))
                or not math.isfinite(captured_at) or captured_at <= self.fresh_after
                or not 0 <= now-captured_at <= self.max_frame_age):
            return None
        candidates = []
        for detection in detections:
            if detection.get('inside_work_area') is not True or detection.get('center_inside_mask') is not True:
                continue
            color, xy, confidence = (detection.get(k) for k in ('color', 'xy_mm', 'confidence'))
            if color not in {'GREEN', 'RED', 'YELLOW'} or not isinstance(xy, (list, tuple)) or len(xy) != 2:
                continue
            values = [*xy, confidence]
            if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in values):
                continue
            if not self.min_confidence <= confidence <= 1:
                continue
            candidates.append((confidence, color, float(xy[0]), float(xy[1])))
        if not candidates:
            return None
        # One target, highest confidence; deterministic tie break by color/XY.
        _, color, x, y = max(candidates)
        self._active = dict(stage_id=uuid.uuid4().hex, x=x, y=y,
                            G=int(color == 'GREEN'), R=int(color == 'RED'), Y=int(color == 'YELLOW'))
        self.state = 'WAIT_ACCEPTED'  # Lock BEFORE the caller transmits this payload.
        self._deadline = now+self.timeout
        return self.active


In [ ]:
# Simulasi murni: tidak membuka TCP/serial dan tidak menggerakkan motor.
import stage_coordinator
importlib.reload(stage_coordinator)
from stage_coordinator import StageCoordinator

demo_time = [100.0]
coordinator = StageCoordinator(clock=lambda: demo_time[0], min_confidence=CONF)
coordinator.on_status({'event': 'READY', 'home': True})
demo_detection = dict(color='RED', confidence=1.0, xy_mm=[10., 550.],
                      inside_work_area=True, center_inside_mask=True)
demo_time[0] = 101.0
first = coordinator.offer([demo_detection], captured_at=101.0)
assert first is not None
coordinator.on_status({'event': 'ACCEPTED', 'stage_id': first['stage_id']})
demo_time[0] = 102.0
another = dict(demo_detection, color='GREEN', xy_mm=[10., 410.])
assert coordinator.offer([another], captured_at=102.0) is None
assert coordinator.on_status('OK') is False
assert coordinator.state == 'BUSY'
demo_time[0] = 110.0
coordinator.on_status({'event': 'DONE', 'stage_id': first['stage_id'], 'home': True})
assert coordinator.offer([another], captured_at=102.0) is None  # Hasil saat stage lama dibuang.
demo_time[0] = 111.0
second = coordinator.offer([another], captured_at=111.0)
assert second is not None and second['stage_id'] != first['stage_id']
print('Simulasi lulus: 1 target aktif, tidak ada antrean saat BUSY, stage berikutnya memakai frame baru.')
STAGE_PROTOCOL = {
    'version': 1, 'transport': 'TCP JSON lines; planned integration', 'port': 5000,
    'cycle': ['pick', 'place_in_firmware_color_bowl', 'home'],
    'states': ['WAIT_READY', 'IDLE', 'WAIT_ACCEPTED', 'BUSY', 'FAULT'],
    'completion': 'DONE matching stage_id AND home=true after ACCEPTED',
    'busy_policy': 'drop detections; no target queue; never replace active target',
    'next_frame': 'callback monotonic timestamp strictly after DONE receipt',
    'automatic_retry': False, 'tcp_adapter_connected': False,
    'firmware_status': 'Terima_data_icamv2.ino only acknowledges receipt; motion cycle hook missing',
}


## 7. Bundle yang dapat diverifikasi

Folder unik per evaluasi mencegah model/report/kalibrasi lama tercampur.
Berisi best.pt, best.onnx, runtime, mapping SmoothingPitch, template kalibrasi,
audit, metrik PT dan ONNX, pemilihan confidence, parity, versi paket, argumen training,
contoh kegagalan dan SHA256SUMS.json. Kekurangan data lapangan tetap tertulis pada readiness.

reports/merge_manifest.json menyimpan asal sampel dan atribusi kedua dataset.
reports/onnx_quality.json/by_source memisahkan evaluasi crop lama dan dataset matras baru.

stage_coordinator.py dan stage_protocol.json menyimpan pengunci satu stage serta kontrak
status Arduino. Belum ada TCP adapter/motor otomatis; simulasi di atas memeriksa urutannya.


In [ ]:
import importlib.metadata

BUNDLE = ARTIFACT_ROOT/('deploy_arm_colors_'+RUN_ID)
BUNDLE.mkdir(exist_ok=False)
for source, name in [(BEST_PT, 'best.pt'), (ONNX_PATH, 'best.onnx'),
                     (ROOT/'icam_color_runtime.py', 'icam_color_runtime.py'),
                     (ROOT/'robot_mapping.py', 'robot_mapping.py'),
                     (ROOT/'arm_quality.py', 'arm_quality.py'),
                     (ROOT/'stage_coordinator.py', 'stage_coordinator.py'),
                     (ONNX_PATH.parent/'runtime_config.json', 'runtime_config.json')]:
    shutil.copy2(source, BUNDLE/name)
(BUNDLE/'labels.txt').write_text('\n'.join(NAMES[i] for i in sorted(NAMES))+'\n')
(BUNDLE/'stage_protocol.json').write_text(json.dumps(STAGE_PROTOCOL, indent=2))
(BUNDLE/'robot_mapping.json').write_text(json.dumps(ROBOT_MAPPING, indent=2))
(BUNDLE/'robot_plane.example.json').write_text(json.dumps(CALIBRATION, indent=2))
if calibration is not None:
    (BUNDLE/'robot_plane.json').write_text(json.dumps(CALIBRATION, indent=2))
reports = BUNDLE/'reports'
reports.mkdir()
for report in EVAL_DIR.glob('*.json'):
    shutil.copy2(report, reports/report.name)
if (EVAL_DIR/'error_examples').is_dir():
    shutil.copytree(EVAL_DIR/'error_examples', reports/'error_examples')
readiness = dict(dataset_coverage=REPORT['coverage'],
                 validation_precision_target_met=THRESHOLD_REPORT['precision_target_met'],
                 camera_mapping_defined=calibration is not None,
                 camera_mapping_physically_verified=False,
                 icam_latency_measured=False, robot_pick_drop_role_confirmed=False,
                 robot_motion_enabled=False, stage_protocol=STAGE_PROTOCOL,
                 status='Model dievaluasi offline; mapping fisik dan uji ICAM masih terpisah')
manifest = dict(created_utc=datetime.now(timezone.utc).isoformat(), run_id=RUN_ID,
                dataset_sha256=DATASET_SHA256,
                source_zips=[{k: s[k] for k in ('source_id', 'zip_name', 'sha256', 'id_remap', 'attribution')}
                             for s in MERGE_MANIFEST['sources']],
                names=NAMES, imgsz=IMGSZ, task='segment',
                preprocessing='stretch_rgb_nchw_fp32', confidence=CONF,
                onnx=dict(opset=13, batch=1, dynamic=False, nms=False),
                runtime='ONNX Runtime CPU; NPU integration not configured',
                firmware='SmoothingPitch.ino', robot_motion_enabled=False,
                training_versions={p: importlib.metadata.version(p) for p in
                                   ['ultralytics', 'torch', 'numpy', 'opencv-python', 'onnx', 'onnxruntime']},
                dataset_audit=REPORT, evaluation=VALIDATION, onnx_quality=ONNX_QUALITY,
                readiness=readiness)
(BUNDLE/'manifest.json').write_text(json.dumps(manifest, indent=2))
(BUNDLE/'requirements-runtime.txt').write_text('numpy<2\nonnxruntime==1.20.1\n')
readme = """ICAM-300 / QCS6490 / SmoothingPitch.ino mapping preview
1. ICAM Python 3.10: use CAMNavi2 and OpenCV from the Advantech BSP.
   Optional venv: python3 -m venv --system-site-packages .venv
   pip install -r requirements-runtime.txt
   These pins are for ICAM/BSP, NOT Colab Python 3.13.
   Use a matching aarch64/Python/glibc ONNX Runtime wheel or vendor build.
2. Verify SHA256SUMS.json before copying parts from another bundle.
   runtime_config.json must match best.onnx SHA256 (checked by ColorModel).
3. Image: python3 icam_color_runtime.py --model best.onnx --source meja.jpg
4. Camera: release camera from vendor web utility per BSP guide:
   sudo systemctl stop web.service
   python3 icam_color_runtime.py --source camnavi --device-name iCam500 --frames 100 --threads 2
   Use actual enumerated device name/resolution. --show needs GUI OpenCV.
   After closing SDK: sudo systemctl start web.service
5. robot_mapping.json records color REFERENCE poses, not confirmed pick/drop locations.
   Firmware accepts six numeric fields for PTP, seven for parabolic; newline terminated.
   HOME moves; ZERO sets the physical upright reference. No XY:/SAVE/POINTS protocol here.
   Servo is written after motion. Firmware has blocking moves and no STOP command.
   This bundle never opens serial. Preview does not certify physical collision clearance.
6. Fill robot_plane.example.json with measured pixel/robot XY correspondences at fixed Z.
   Save as robot_plane.json. Validate independent points in mm with the fixed camera mount.
   python3 icam_color_runtime.py --source camnavi --calibration robot_plane.json
   Output is XY preview only; no automatic motion or inferred depth.
   stage_coordinator.py provides a tested single-stage gate, not a TCP/motor adapter.
   See stage_protocol.json: ACCEPTED means received; matching DONE + home=true ends the stage.
   Terima_data_icamv2.ino currently replies only OK and has no connected motion cycle.
   Do not treat OK as completion. Discard busy-period detections and read a new frame after DONE.
7. reports/onnx_quality.json measures the deployed decoder at the chosen confidence.
   Confidence selection uses valid, not test. Crops without negatives cannot measure table false alarms.
   Colab CPU timing is not ICAM FPS. Test full scenes, empty table, hands, shadows and glare.
   ONNX is FP32 CPU. NPU requires separate conversion and accuracy/latency validation.
"""
(BUNDLE/'README-ICAM.txt').write_text(readme, encoding='utf-8')
checksums = {str(p.relative_to(BUNDLE)).replace('\\', '/'): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in sorted(BUNDLE.rglob('*')) if p.is_file()}
(BUNDLE/'SHA256SUMS.json').write_text(json.dumps(checksums, indent=2))
BUNDLE_ZIP = Path(shutil.make_archive(str(BUNDLE), 'zip', BUNDLE))
print('Bundle:', BUNDLE_ZIP)
print(json.dumps(readiness, indent=2))
display(__import__('IPython').display.FileLink(str(BUNDLE_ZIP)))


In [ ]:
# Opsional: unduh bundle dari Colab setelah training/ekspor selesai.
DOWNLOAD_BUNDLE = False
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE_ZIP))

## 8. Menjalankan hasil di ICAM-300

Ekstrak bundle dan ikuti `README-ICAM.txt`. Training tidak perlu diulang di kamera.
Periksa import CAMNavi2/OpenCV bawaan BSP; pasang ONNX Runtime yang sesuai Python/aarch64/glibc.
Panduan lokal menyebut Yocto Linux + Python 3.10. Dukungan wheel Python bergantung image BSP.
Jika tidak cocok, gunakan build vendor atau jalankan runtime pada PC yang menerima file/stream ICAM.
Jangan mengasumsikan ICAM muncul sebagai webcam USB index 0.

```bash
# Hanya di ICAM, sesuai Programming Guide bab 3; hentikan pemilik kamera sebelumnya.
sudo systemctl stop web.service
python3 icam_color_runtime.py --source camnavi --device-name iCam500 --frames 100
# Untuk pratinjau GUI lokal, tambah --show; mode default headless menghasilkan JSON.
# Setelah runtime ditutup, bila ingin memakai web utility kembali:
sudo systemctl start web.service
```

Panduan vendor menggunakan nama device lama `iCam500` pada contoh ICAM-300.
Runtime menampilkan hasil enumerasi; sesuaikan `--device-name` dan resolusi dengan SDK sebenarnya.
Ia memakai callback JPEG warna, satu frame terbaru, tanpa antrean frame lama.
Eksposur/white balance/pencahayaan harus konsisten dengan data training. Jangan grayscale frame.

Uji ketiga kelas bersamaan, beberapa benda dengan warna sama, meja kosong, tangan/robot dalam gambar,
bayangan, dan pantulan. Catat false positive, objek terlewat, error pusat XY (mm), serta latency
di ICAM. Tidak ada angka FPS/akurasi hardware yang dijanjikan notebook ini.

Referensi:

- [Google Colab FAQ: runtime, GPU, dan penyimpanan Drive](https://research.google.com/colaboratory/faq.html)
- [Wheel ONNX 1.18.0 untuk Python 3.13](https://pypi.org/project/onnx/1.18.0/#files), [NumPy 2.2.6](https://pypi.org/project/numpy/2.2.6/#files)
- [Advantech ICAM-300: Qualcomm QCS6490](https://www.advantech.com/en-sg/products/ce666c81-b9fc-4675-b7aa-0c16ce758636/icam-300/mod_c348d281-94f9-4df7-82cb-cd9e35388c74)
- `Programming Guide for ICAM-300.pdf` lokal: tabel 1.1, bab 3, contoh CAMNavi2 bab 5 (hal. 54–56).
- `ICAM-300_User_Manual_Ed.2.pdf` lokal dan `SmoothingPitch.ino` yang Anda kirim sebagai acuan mapping/protokol.
- [BCN3D Moveo](https://github.com/BCN3D/BCN3D-Moveo)
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/), [segmentation](https://docs.ultralytics.com/tasks/segment/), [ONNX](https://docs.ultralytics.com/integrations/onnx/)
- [Konfigurasi Ultralytics v8.3.221 yang dipakai](https://github.com/ultralytics/ultralytics/blob/v8.3.221/ultralytics/cfg/default.yaml)
- [Qualcomm QIM pipeline dan postprocessing](https://www.qualcomm.com/developer/blog/2024/07/qualcomm-linux-sample-apps-ai-object-detection-parallel-ai)
- [Dataset Roboflow v5, CC BY 4.0](https://universe.roboflow.com/ahmad-zuhril-fahrizal/arm-warna-3/dataset/5)
